#  Goal: Encryption of a PDF file with KEM and AES
In this task, a combination of two cryptographic methods shall be used:

- KEM (Key Encapsulation Mechanism):
    Serves the secure and asymmetric distribution of a secret key.

- AES (Advanced Encryption Standard):
    A symmetric encryption algorithm that encrypts the actual content (e.g., the PDF file).

The goal is to use **KEM** to securely exchange a shared **symmetric key** between sender (Alice) and receiver (Bob),
in order to subsequently encrypt the PDF file efficiently with **AES** .


siehe:  https://github.com/open-quantum-safe/liboqs-python/blob/main/examples/kem.py  
https://github.com/open-quantum-safe/liboqs-python/blob/main/oqs/oqs.py  
https://www.pycrypto.org/api/current/Crypto.Cipher.AES-module.html#new  
https://pycryptodome.readthedocs.io/en/latest/src/cipher/modern.html#eax-mode  

![alternativ](KEM_AES1.png)

  

# Encryption

In [3]:
from Crypto.Cipher import AES
import oqs
import os

# Pfad zur PDF-Datei
input_file_path = "messageToReceiver.pdf"

# 1. pdf-Datei lesen
with open(input_file_path, "rb") as f:
    plaintext = f.read()

# 2. Empfänger erzeugt Kyber512-Schlüsselpaar
# Sie erstellt ein KEM-Objekt als Receiver für den Algorithmus Kyber512 aus der oqs-Bibliothek.
receiver = oqs.KeyEncapsulation("Kyber512")

#Generate a new keypair and returns the public key.
#If needed, the secret key can be obtained with export_secret_key().
#erzeugt Public/Private Key (private intern gespeichert) und gibt Public key zurück
public_key = receiver.generate_keypair()

# 3. Sender verschlüsselt ein Secret mit dem Public Key
# Sie erstellt ein KEM-Objekt als Sender für den Algorithmus Kyber512 aus der oqs-Bibliothek.
sender = oqs.KeyEncapsulation("Kyber512")
#Generate and encapsulates a secret using the provided public key.
#and return ciphertext and secret
ciphertext_kem, shared_secret_sender = sender.encap_secret(public_key)

# 4. AES-Schlüssel aus shared_secret ableiten
aes_key = shared_secret_sender[:16]  # AES-128

# 5. AES-Verschlüsselung (EAX-Modus)
# Create a new AES cipher Crypto.Cipher.AES.new(key, mode, *args, **kwargs)
# param: key(The secret key to use in the symmetric cipher), mode(The chaining mode to use for encryption or decryption)
# Keyword Arguments: nonce(A value that must never be reused for any other encryption done with this key) For MODE_EAX, MODE_GCM and MODE_SIV there are no restrictions on its length (recommended: 16 bytes)
# EAX ist ein sogenannter AEAD-Modus (Authenticated Encryption with Associated Data):
# AES in EAX-Modus verschlüsselt die Daten und erzeugt gleichzeitig einen MAC (Message Authentication Code).
aes_cipher_sender = AES.new(aes_key, AES.MODE_EAX)
nonce = aes_cipher_sender.nonce

#encrypt_and_digest(plaintext, output=None) Perform encrypt() and digest() in one go.
#returns a tupel with two items: {ciphertext, MAC tag(Message Authentication Code)}
#MAC tag = Message Authentication Code → stellt sicher, dass Daten authentisch und unverändert sind
ciphertext_doc, tag = aes_cipher_sender.encrypt_and_digest(plaintext)

# 6. Speichern der verschlüsselten Daten
os.makedirs("encrypted", exist_ok=True)
with open("encrypted/ciphertext_doc.bin", "wb") as f: f.write(ciphertext_doc)
with open("encrypted/ciphertext_kem.bin", "wb") as f: f.write(ciphertext_kem)
with open("encrypted/aes_nonce.bin", "wb") as f: f.write(nonce)
with open("encrypted/aes_tag.bin", "wb") as f: f.write(tag)

print("Encryption completed :) ")


Verschlüsselung abgeschlossen :) 


# Decryption 

In [6]:
# 1. Dateien laden
with open("encrypted/ciphertext_doc.bin", "rb") as f: ciphertext_doc = f.read()
with open("encrypted/ciphertext_kem.bin", "rb") as f: ciphertext_kem = f.read()
with open("encrypted/aes_nonce.bin", "rb") as f: nonce = f.read()
with open("encrypted/aes_tag.bin", "rb") as f: tag = f.read()

# 2. Empfänger entschlüsselt shared_secret
#Decapsulate the ciphertext and returns the secret.
shared_secret_receiver = receiver.decap_secret(ciphertext_kem)
aes_key_recv = shared_secret_receiver[:16]

# 3. AES-Entschlüsselung
aes_cipher_recv = AES.new(aes_key_recv, AES.MODE_EAX, nonce=nonce)
#decrypt_and_verify(ciphertext, mac_tag, output=None) Perform decrypt() and verify() in one go.
# returns plaintext, in ValueError case: – if the MAC tag is not valid, that is, if the entire message should not be trusted.
decrypted_plaintext = aes_cipher_recv.decrypt_and_verify(ciphertext_doc, tag)

# 4. Speichern als PDF
with open("decrypted_message.pdf", "wb") as f:
    f.write(decrypted_plaintext)

print("Decryption completed :) ")


ValueError: MAC check failed

# die Verschlüsselte und Entschlüsselte PDF- Datei vergleichen

In [20]:
# Original und entschlüsselte Datei laden
with open("messageToReceiver.pdf", "rb") as f:
    original = f.read()

with open("decrypted_message.pdf", "rb") as f:
    decrypted = f.read()
# with open("U05.pdf", "rb") as f:
#    decrypted = f.read()

# Vergleich
if original == decrypted:
    print("The decrypted file is IDENTICAL to the original :) ")
else:
    print("The decrypted file is NOT identical :( ")
    print(f"Originalgröße: {len(original)} Bytes")
    print(f"Entschlüsselt: {len(decrypted)} Bytes")


Die entschlüsselte Datei ist IDENTISCH mit dem Original :) 
